In [1]:
import torch
import numpy as np
import timm
import mlp
import dataloader as dt
import utils

torch.set_printoptions(sci_mode=False)
device = 'cuda'

In [2]:
model = timm.create_model('inception_v3', pretrained=True)
model.eval()
model = model.to(device)

loss_fn = torch.nn.MSELoss()

model_mlp = mlp.MLP(8217, 2048).to(device)
model_mlp_dir = 'lr-models/model_inception.pt'
model_mlp.load_state_dict(torch.load(model_mlp_dir))
model_mlp.eval()

MLP(
  (model): Sequential(
    (0): Linear(in_features=8217, out_features=8217, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=8217, out_features=8217, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.1, inplace=False)
    (6): Linear(in_features=8217, out_features=2048, bias=True)
  )
)

In [3]:
layers = []
for name, module in model.named_modules():
	#print(name) # names of all layers
	if "conv" in name or "head_drop" == name:
		#print(f"Module hook registered for: {name} # names of tracked layers
		module.register_forward_hook(lambda m, i, o: layers.append(o))

In [4]:
test_loader = dt.get_loader(split='test', resize_dim=299, shuffle=False, batch=1)
img_indexes = [0, 500, 1500, 7500, 9500]
for i in img_indexes:
	img, lbl = test_loader.dataset[i]

	img = torch.unsqueeze(img.to(torch.float32).to(device), dim=0)
	lbl = torch.from_numpy(np.array([lbl])).type(torch.long).to(device)

	outputs = model(img)
	_, pre = torch.max(outputs.data, 1)

	# Check if the initial prediction is correct, this should print True
	print(pre == lbl)


	# Get the LR score for clean sample
	f1 = layers[15][:, :3].view(1, 3 * 35 * 35)
	f2 = layers[25][:, :3].view(1, 3 * 35 * 35)
	f3 = layers[35][:, :3].view(1, 3 * 17 * 17)
	f_tot = torch.cat([f1, f2, f3], dim=1)

	o2 = layers[-1]

	mlp_out = model_mlp(f_tot)
	lr_score_clean = loss_fn(mlp_out, o2)

	# Attack and get the LR score for attacked sample
	adv_img = utils.get_attack(img, lbl, model, 'pgd')

	del layers
	layers = []

	outputs_adv = model(adv_img)
	_adv, pre_adv = torch.max(outputs_adv.data, 1)

	# Check if attack successful, this should print False
	print(pre_adv == lbl)

	f1 = layers[15][:, :3].view(1, 3 * 35 * 35)
	f2 = layers[25][:, :3].view(1, 3 * 35 * 35)
	f3 = layers[35][:, :3].view(1, 3 * 17 * 17)
	f_tot = torch.cat([f1, f2, f3], dim=1)

	o2 = layers[-1]

	mlp_out = model_mlp(f_tot)
	lr_score_adv = loss_fn(mlp_out, o2)

	del layers
	layers = []

	print('LR score for clean: {} --- LR score for attacked sample: {}'.format(lr_score_clean.item(), lr_score_adv.item()))
	print()

tensor([True], device='cuda:0')
tensor([False], device='cuda:0')
LR score for clean: 0.06722833216190338 --- LR score for attacked sample: 0.5542400479316711

tensor([True], device='cuda:0')
tensor([False], device='cuda:0')
LR score for clean: 0.06797292828559875 --- LR score for attacked sample: 0.4202122390270233

tensor([True], device='cuda:0')
tensor([False], device='cuda:0')
LR score for clean: 0.08431819081306458 --- LR score for attacked sample: 0.20681972801685333

tensor([True], device='cuda:0')
tensor([False], device='cuda:0')
LR score for clean: 0.06743744015693665 --- LR score for attacked sample: 0.2678071856498718

tensor([True], device='cuda:0')
tensor([False], device='cuda:0')
LR score for clean: 0.09039591252803802 --- LR score for attacked sample: 0.7809993624687195

